In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torchao>=0.16.0'],
    check=True,
)

import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
import diffusers
print(f'diffusers: {diffusers.__version__}')
import peft
print(f'peft: {peft.__version__}')

# peft 0.19 can raise on old torchao even after upgrade; force standard LoRA path
import peft.import_utils as _peft_iu
import peft.tuners.lora.torchao as _peft_torchao
_peft_iu.is_torchao_available = lambda: False
_peft_torchao.is_torchao_available = lambda: False

import json
from pathlib import Path

def _sprite_dirs(root):
    dirs = []
    for p in root.rglob('sprites'):
        if p.is_dir():
            n = len(list(p.glob('*.png')))
            if n > 0:
                dirs.append((n, p))
    return dirs

candidates = _sprite_dirs(Path('/kaggle/input'))
assert candidates, 'No sprite PNG directory found under /kaggle/input'
DATASET_DIR = max(candidates, key=lambda t: t[0])[1]
print(f'Sprites: {DATASET_DIR} ({len(list(DATASET_DIR.glob("*.png")))} PNGs)')
meta = DATASET_DIR / 'metadata.jsonl'
print(f'metadata.jsonl: {meta.exists()}')

sd_dirs = list(Path('/kaggle/input').glob('stable-diffusion-1-5*'))
assert sd_dirs, 'SD 1.5 dataset not found'
SD_DIR = sd_dirs[0]
ckpt_files = list(SD_DIR.rglob('*.ckpt'))
assert ckpt_files, 'No .ckpt under SD dataset'
CKPT = str(ckpt_files[0])
print(f'SD ckpt: {CKPT}')


In [ ]:
# === LOAD SD1.5 FROM SINGLE CKPT (proper UNet/VAE/TE/tokenizer) ===
from diffusers import (
    StableDiffusionPipeline,
    DDPMScheduler,
    DDIMScheduler,
)
from peft import LoraConfig, get_peft_model_state_dict
from peft.utils import set_peft_model_state_dict

# torch 2.10 defaults weights_only=True; classic .ckpt needs full unpickle
_orig_torch_load = torch.load
def _torch_load_full(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_full

pipe = StableDiffusionPipeline.from_single_file(
    CKPT,
    torch_dtype=torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
)
torch.load = _orig_torch_load
pipe = pipe.to('cuda')

unet = pipe.unet
vae = pipe.vae
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer

# train with DDPM noise schedule (SD1.5 defaults)
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

for m in [unet, vae, text_encoder]:
    m.requires_grad_(False)
    m.eval()

unet.add_adapter(LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'],
    lora_dropout=0.05,
    bias='none',
))

trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
total = sum(p.numel() for p in unet.parameters())
print(f'Model loaded. Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')


In [ ]:
# === DATALOADER (256 for small GBA sprites; NEAREST keep pixels) ===
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

RESOLUTION = 256

class SpriteDataset(Dataset):
    def __init__(self, dataset_dir, resolution=256):
        self.metadata = []
        meta_path = dataset_dir / 'metadata.jsonl'
        if meta_path.exists():
            with open(meta_path) as f:
                for line in f:
                    self.metadata.append(json.loads(line))
        else:
            for img_path in sorted(dataset_dir.glob('*.png')):
                self.metadata.append({
                    'file_name': img_path.name,
                    'caption': 'pixel art sprite, game character, gba style, white background',
                })
        self.dataset_dir = dataset_dir
        self.transform = transforms.Compose([
            transforms.Resize((resolution, resolution), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        e = self.metadata[idx]
        img = Image.open(self.dataset_dir / e['file_name']).convert('RGB')
        caption = e.get('caption', 'pixel art sprite, gba style')
        # keep training captions closer to inference style
        if 'white background' not in caption:
            caption = caption + ', white background'
        return {'image': self.transform(img), 'caption': caption}

dataset = SpriteDataset(DATASET_DIR, RESOLUTION)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2, drop_last=True)
print(f'Sprites: {len(dataset)} | res={RESOLUTION} | batch=2')
print('sample caption:', dataset[0]['caption'])


In [ ]:
# === TRAIN ===
import torch.nn.functional as F
from tqdm.auto import tqdm
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

LR, STEPS, SAVE_EVERY, GRAD_ACCUM = 1e-4, 3000, 500, 2
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=LR,
    weight_decay=0.01,
)
unet.train()
vae.eval()
text_encoder.eval()

step, losses = 0, []
pbar = tqdm(total=STEPS, desc='Training')
while step < STEPS:
    for batch in dataloader:
        if step >= STEPS:
            break
        images = batch['image'].to('cuda', dtype=torch.float32)
        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * vae.config.scaling_factor
            tokens = tokenizer(
                batch['caption'],
                padding='max_length',
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors='pt',
            ).input_ids.to('cuda')
            ehs = text_encoder(tokens)[0]
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        ts = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device='cuda').long()
        noisy = noise_scheduler.add_noise(latents, noise, ts)
        pred = unet(noisy, ts, ehs).sample
        if noise_scheduler.config.prediction_type == 'epsilon':
            target = noise
        elif noise_scheduler.config.prediction_type == 'v_prediction':
            target = noise_scheduler.get_velocity(latents, noise, ts)
        else:
            raise ValueError(noise_scheduler.config.prediction_type)
        loss = F.mse_loss(pred.float(), target.float()) / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, unet.parameters()), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        losses.append(loss.item() * GRAD_ACCUM)
        pbar.set_postfix(loss=f'{losses[-1]:.4f}')
        pbar.update(1)
        step += 1
        if step % SAVE_EVERY == 0:
            print(f'\nStep {step}: avg loss {sum(losses[-SAVE_EVERY:]) / SAVE_EVERY:.4f}')
pbar.close()
print(f'\nDone. Final loss: {sum(losses[-100:]) / min(100, len(losses)):.4f}')


In [ ]:
# === SAVE LORA + GENERATE WITH DDIM ===
from peft import get_peft_model_state_dict
from safetensors.torch import save_file
from PIL import Image as PILImage

SAVE_DIR = Path('/kaggle/working/spritelab_lora')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

lora_state = get_peft_model_state_dict(unet)
save_file(lora_state, SAVE_DIR / 'pytorch_lora_weights.safetensors')
with open(SAVE_DIR / 'adapter_config.json', 'w') as f:
    json.dump({
        'r': 16,
        'lora_alpha': 16,
        'target_modules': ['to_q', 'to_k', 'to_v', 'to_out.0'],
        'base': 'sd1.5',
        'resolution': RESOLUTION,
        'steps': STEPS,
    }, f, indent=2)
print(f'Saved LoRA ({len(lora_state)} tensors) to {SAVE_DIR}')

unet.eval()
pipe.unet = unet
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.safety_checker = None

def pixelate(img, sz=64):
    return img.resize((sz, sz), PILImage.NEAREST).resize(img.size, PILImage.NEAREST)

def quantize(img, c=32):
    return img.convert('P', palette=PILImage.ADAPTIVE, colors=c).convert('RGB')

prompts = [
    'pixel art sprite, fire mage character, gba style, white background',
    'pixel art sprite, knight with sword, gba style, white background',
    'pixel art sprite, blue dragon, gba style, white background',
    'pixel art sprite, female archer, gba style, white background',
]

generator = torch.Generator(device='cuda').manual_seed(42)
for i, p in enumerate(prompts):
    out = pipe(
        p,
        num_inference_steps=40,
        guidance_scale=7.5,
        width=RESOLUTION,
        height=RESOLUTION,
        generator=generator,
    ).images[0]
    out.save(f'/kaggle/working/sprite_{i:02d}_raw.png')
    img = quantize(pixelate(out, 64))
    img.save(f'/kaggle/working/sprite_{i:02d}.png')
    print(f'saved sprite_{i:02d}.png | {p}')

evaluation_prompts = [
    'pixel art sprite, fire mage character, gba style, white background',
    'pixel art sprite, knight with sword, gba style, white background',
    'pixel art sprite, blue dragon, gba style, white background',
    'pixel art sprite, female archer, gba style, white background',
    'pixel art sprite, green slime monster, gba style, white background',
    'pixel art sprite, red armored robot, gba style, white background',
    'pixel art sprite, wooden treasure chest, gba style, white background',
    'pixel art sprite, glowing blue sword, gba style, white background',
    'pixel art sprite, yellow airship, gba style, white background',
    'pixel art sprite, stone castle tower, gba style, white background',
    'pixel art sprite, purple magic potion, gba style, white background',
    'pixel art sprite, small brown dog, gba style, white background',
]
evaluation_seeds = [42, 31415]
evaluation_dir = Path('/kaggle/working/evaluation')
evaluation_dir.mkdir(parents=True, exist_ok=True)
evaluation_manifest = []
evaluation_images = []

for prompt_index, prompt in enumerate(evaluation_prompts):
    for seed in evaluation_seeds:
        out = pipe(
            prompt,
            num_inference_steps=40,
            guidance_scale=7.5,
            width=RESOLUTION,
            height=RESOLUTION,
            generator=torch.Generator(device='cuda').manual_seed(seed),
        ).images[0]
        stem = f'eval_{prompt_index:02d}_seed_{seed}'
        out.save(evaluation_dir / f'{stem}_raw.png')
        processed = quantize(pixelate(out, 64))
        processed.save(evaluation_dir / f'{stem}.png')
        evaluation_images.append(processed)
        evaluation_manifest.append({'file': f'{stem}.png', 'prompt': prompt, 'seed': seed})

with open(evaluation_dir / 'manifest.json', 'w') as manifest_file:
    json.dump(evaluation_manifest, manifest_file, indent=2)

grid_columns = 4
grid_rows = (len(evaluation_images) + grid_columns - 1) // grid_columns
grid = PILImage.new('RGB', (grid_columns * RESOLUTION, grid_rows * RESOLUTION), 'white')
for image_index, image in enumerate(evaluation_images):
    x = image_index % grid_columns * RESOLUTION
    y = image_index // grid_columns * RESOLUTION
    grid.paste(image, (x, y))
grid.save('/kaggle/working/evaluation_grid.png')
print(f'Saved fixed evaluation grid with {len(evaluation_images)} images')

print('\nDone! Download spritelab_lora/, evaluation/, and evaluation_grid.png from Output.')
